In [ ]:
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import numqi

from utils import HilbertSchmidtMeasure

In [ ]:
def optimal_witness_isostropic(local_dim, schmidt_number):
    # Construct the optimal geometric Schmidt witness for the isotropic state
    tmp0 = numqi.state.maximally_entangled_state(local_dim)
    tmp1 = (schmidt_number - 1) / local_dim * np.eye(local_dim**2) - tmp0[:,np.newaxis] @ tmp0[np.newaxis,:]
    ret = (local_dim/np.sqrt(local_dim**2-1))*tmp1
    return ret

In [ ]:
hf_F_to_alpha = lambda F, d: (F*(d**2)-1)/(d**2-1)

dim = 4
k_list = [2, 3, 4]
F_list = np.linspace(0, 1, 20)
alpha_list = [hf_F_to_alpha(F, dim) for F in F_list]

ret_list = [[] for _ in k_list]
for k in k_list:
    witness = optimal_witness_isostropic(dim, k)
    for alpha in alpha_list:
        rho = numqi.state.Isotropic(dim, alpha)
        ret = np.real(np.trace(rho @ witness))
        ret_list[k-2].append(ret)

fig, ax = plt.subplots()
ax.plot(F_list, ret_list[0], label=r'$k=2$', marker='o')
ax.plot(F_list, ret_list[1], label=r'$k=3$', marker='o')
ax.plot(F_list, ret_list[2], label=r'$k=4$', marker='o')
ax.hlines(0, 0, 1, colors='k', linestyles='dashed')
ax.text(0.8, 0.3, 'SEP', ha='center', va='center', size=12)
ax.text(0.2, -0.3, 'ENT', ha='center', va='center', size=12)
ax.set_xlabel(r'$F$')
ax.set_ylabel(r'$\text{Tr}[\rho W]$')
ax.legend()

In [ ]:
def depolarizing_channel(rho, p):
    dim = rho.shape[0]
    return p * rho + (1-p) * np.eye(dim) / dim

In [ ]:
_, rho_bes = numqi.entangle.load_upb('tiles', return_bes=True)
model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=36)
model.set_target_rho(rho_bes)
theta_optim = numqi.optimize.minimize(model, num_repeat=10, tol=1e-14, print_every_round=0)
info = model(return_info=True)[1]
print(info['distance'])

In [ ]:
sigma = info['sigma']
threshold = 1e-8
real_part = np.real(sigma)
imag_part = np.imag(sigma)
real_part[np.abs(real_part)<threshold] = 0
imag_part[np.abs(imag_part)<threshold] = 0
sigma_re = real_part + 1j*imag_part
print(sigma_re)

In [ ]:
ret_list = []
sigma_list = []
p_list = np.linspace(0.8, 1, 10)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_bes, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    ret_list.append(info['distance'])
    sigma_list.append(info['sigma'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, ret_list, marker='o')
ax.set_xlabel(r'$p$')
ax.vlines(0.8908, 0, 0.04, colors='k', linestyles='dashed')
ax.set_ylabel(r'HS measure')